# PNR Polarisation: 2-Layer NF Inference

Test the 2-layer normalizing-flow checkpoint on `00058_du.dat` and `00058_uu.dat`.

The attached fit notebook models air/Pt/Co/Al2O3. Here Pt and Co are the two inferred layers, and each polarisation channel is tested independently with its effective Co SLD. This is a model smoke test, not a joint magnetic PNR fit.

## Setup

In [ ]:
%matplotlib inline

from pathlib import Path
import os
import sys

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import matplotlib as mpl

mpl.use("module://matplotlib_inline.backend_inline", force=True)

import matplotlib.pyplot as plt
import numpy as np
import torch

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "exp_data" / "00058_du.dat").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError("Run this notebook from inside the inverse-eval repo.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from device_utils import detect_torch_device, summarize_torch_backends
from nf_statistics import compute_nf_sample_statistics
from reflectorch import EasyInferenceModel


def to_numpy(value):
    if torch.is_tensor(value):
        return value.detach().cpu().numpy()
    return np.asarray(value)


plt.rcParams["text.usetex"] = False
torch.manual_seed(42)

print(f"Project root    : {PROJECT_ROOT}")
print(f"PyTorch version : {torch.__version__}")
print(f"Backends        : {summarize_torch_backends()}")

## Configuration

In [ ]:
DATA_FILES = {
    "du": PROJECT_ROOT / "exp_data" / "00058_du.dat",
    "uu": PROJECT_ROOT / "exp_data" / "00058_uu.dat",
}
FIT_NOTEBOOK = PROJECT_ROOT / "exp_data" / "NR_XRR_Fit_AlO3_Co_Pt.ipynb"
VENDOR_ROOT = PROJECT_ROOT / "vendor" / "nflows_reflectorch"

NF_CONFIG = "example_nf_config_reflectorch_L2.yaml"
NF_SAMPLES = 256
DEVICE = detect_torch_device()
ENABLE_ERROR_BARS_FILTERING = False  # False matches the NR_XRR fit notebook's full positive-Q NR curves.
FILTER_THRESHOLD = 0.3
FILTER_CONSECUTIVE = 3

# Values read from NR_XRR_Fit_AlO3_Co_Pt.ipynb, in 10^-6 A^-2.
PT_SLD = 6.24996
CO_NUCLEAR_SLD = 2.26454
CO_MAG_SLD_BOUNDS = (3.331345, 4.331345)
AL2O3_SLD = 5.71528
FIXED_SLD_HALF_WIDTH = 0.005

PARAM_ORDER = [
    "Pt thickness",
    "Co thickness",
    "air/Pt roughness",
    "Pt/Co roughness",
    "Co/Al2O3 roughness",
    "Pt SLD",
    "effective Co SLD",
    "Al2O3 SLD",
    "r_scale",
    "log10_background",
]

FIT_REFERENCE = {
    "du": {
        "Pt thickness": 43.98,
        "Co thickness": 95.05,
        "air/Pt roughness": 8.12,
        "Pt/Co roughness": 6.83,
        "Co/Al2O3 roughness": 4.46,
        "Pt SLD": 6.25,
        "effective Co SLD": -1.75,
        "Al2O3 SLD": 5.72,
    },
    "uu": {
        "Pt thickness": 43.98,
        "Co thickness": 95.05,
        "air/Pt roughness": 8.12,
        "Pt/Co roughness": 6.83,
        "Co/Al2O3 roughness": 4.46,
        "Pt SLD": 6.25,
        "effective Co SLD": 6.28,
        "Al2O3 SLD": 5.72,
    },
}


def fixed_sld_bounds(center):
    return (center - FIXED_SLD_HALF_WIDTH, center + FIXED_SLD_HALF_WIDTH)


def co_sld_bounds(polarisation):
    sign = {"du": -1.0, "uu": 1.0}[polarisation]
    values = [CO_NUCLEAR_SLD + sign * value for value in CO_MAG_SLD_BOUNDS]
    return (min(values), max(values))


def prior_bounds_for(polarisation):
    structural_bounds = [
        (10.0, 100.0),
        (50.0, 250.0),
        (0.0, 10.0),
        (0.0, 10.0),
        (0.0, 10.0),
        fixed_sld_bounds(PT_SLD),
        co_sld_bounds(polarisation),
        fixed_sld_bounds(AL2O3_SLD),
    ]
    return [*structural_bounds, (0.9, 1.1), (-10.0, -4.0)]


print(f"Fit notebook : {FIT_NOTEBOOK}")
print(f"NF config    : {NF_CONFIG}")
print(f"Device       : {DEVICE}")
print(f"Samples      : {NF_SAMPLES}")
for name in DATA_FILES:
    print(f"{name} prior dim : {len(prior_bounds_for(name))}")

## Load Curves

In [ ]:
def load_curve(path):
    data = np.loadtxt(path)
    if data.ndim != 2 or data.shape[1] < 4:
        raise ValueError(f"Expected four columns Q, R, dR, dQ in {path}")

    q, r, dr, dq = data[:, 0], data[:, 1], data[:, 2], data[:, 3]
    mask = (
        np.isfinite(q)
        & np.isfinite(r)
        & np.isfinite(dr)
        & np.isfinite(dq)
        & (q > 0)
        & (r > 0)
        & (dr > 0)
        & (dq >= 0)
    )
    return {
        "q": q[mask],
        "r": r[mask],
        "dr": dr[mask],
        "dq": dq[mask],
        "raw_points": len(data),
        "removed_points": int((~mask).sum()),
    }


curves = {name: load_curve(path) for name, path in DATA_FILES.items()}

for name, curve in curves.items():
    print(f"{name}: {len(curve['q'])}/{curve['raw_points']} points kept")
    print(f"  Q range    : {curve['q'].min():.5f} - {curve['q'].max():.5f} A^-1")
    print(f"  R range    : {curve['r'].min():.3e} - {curve['r'].max():.3e}")
    print(f"  dR/R range : {(curve['dr'] / curve['r']).min():.3f} - {(curve['dr'] / curve['r']).max():.3f}")
    print(f"  dQ/Q med.  : {np.median(curve['dq'] / curve['q']):.4f}")
    print(f"  removed    : {curve['removed_points']}")

fig, ax = plt.subplots(figsize=(6, 4), constrained_layout=True)
for name, curve in curves.items():
    ax.errorbar(curve["q"], curve["r"], yerr=curve["dr"], fmt=".", ms=3, alpha=0.65, label=name)
ax.set(xlabel="Q (A^-1)", ylabel="Reflectivity", yscale="log")
ax.legend(title="polarisation")
plt.show()

## Load Model

In [ ]:
model = EasyInferenceModel(
    config_name=NF_CONFIG,
    root_dir=str(VENDOR_ROOT),
    repo_id=None,
    device=DEVICE,
)

expected_dim = model.trainer.loader.prior_sampler.param_dim
for name in curves:
    assert len(prior_bounds_for(name)) == expected_dim, (name, len(prior_bounds_for(name)), expected_dim)

print(f"Model prior dim: {expected_dim}")

## Run Inference

In [ ]:
def run_channel(name, curve):
    prior_bounds = prior_bounds_for(name)
    prediction = model.preprocess_and_sample(
        reflectivity_curve=curve["r"],
        q_values=curve["q"],
        sigmas=curve["dr"],
        q_resolution=curve["dq"],
        prior_bounds=prior_bounds,
        num_samples=NF_SAMPLES,
        sampling_batch_size=128,
        maximum_sim_batch_size=128,
        calc_sampled_curves=True,
        calc_sampled_sld_profiles=True,
        calc_log_likelihoods=True,
        enable_importance_sampling=True,
        clip_prediction=True,
        enable_error_bars_filtering=ENABLE_ERROR_BARS_FILTERING,
        filter_threshold=FILTER_THRESHOLD,
        filter_consecutive=FILTER_CONSECUTIVE,
    )
    return {"prediction": prediction, "prior_bounds": prior_bounds}


results = {name: run_channel(name, curve) for name, curve in curves.items()}

for name, result in results.items():
    prediction = result["prediction"]
    print(f"{name}: params {to_numpy(prediction['predicted_params_array']).shape}, curves {to_numpy(prediction['sampled_curves']).shape}")

## MAP Estimates

In [ ]:
def summarize_result(name, result):
    prediction = result["prediction"]
    prior_bounds = result["prior_bounds"]
    log_likelihoods = to_numpy(prediction["log_likelihoods"])
    samples = to_numpy(prediction["predicted_params_array"])
    best_idx = int(np.argmax(log_likelihoods))
    map_params = samples[best_idx]
    stats = compute_nf_sample_statistics(samples, log_likelihoods)

    interp_r = to_numpy(prediction["reflectivity_curve_interp"])
    interp_dr = to_numpy(prediction.get("sigmas_interp", interp_r * 0.2 + 1e-12))
    map_curve = to_numpy(prediction["sampled_curves"])[best_idx]
    weighted_mse = float(np.mean(((interp_r - map_curve) / interp_dr) ** 2))

    result.update(best_idx=best_idx, map_params=map_params, weighted_mse=weighted_mse)

    print(f"\n{name} MAP index: {best_idx}")
    print(f"{name} MAP loglike: {log_likelihoods[best_idx]:.4f}")
    print(f"{name} weighted MSE: {weighted_mse:.4g}")
    if "ess" in prediction:
        print(f"{name} ESS: {prediction['ess']:.1f} / {NF_SAMPLES}")
    print(f"{'Parameter':<22} {'Prior min':>11} {'MAP':>12} {'Mean':>12} {'Std':>12} {'Prior max':>11}")
    print("-" * 86)
    for i, label in enumerate(PARAM_ORDER):
        lo, hi = prior_bounds[i]
        print(
            f"{label:<22} {lo:>11.4f} {map_params[i]:>12.4f} "
            f"{stats['nf_params_mean'][i]:>12.4f} {stats['nf_params_std'][i]:>12.4f} {hi:>11.4f}"
        )

    print(f"\n{name} MAP vs NR_XRR fit notebook:")
    print(f"{'Parameter':<22} {'NF MAP':>12} {'NR_XRR fit':>12} {'Delta':>12}")
    print("-" * 62)
    for i, label in enumerate(PARAM_ORDER[:8]):
        fit_value = FIT_REFERENCE[name][label]
        print(f"{label:<22} {map_params[i]:>12.4f} {fit_value:>12.4f} {map_params[i] - fit_value:>12.4f}")


for name, result in results.items():
    summarize_result(name, result)

## Plot Fits and SLD Profiles

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7), constrained_layout=True)

for row, name in enumerate(["du", "uu"]):
    curve = curves[name]
    result = results[name]
    prediction = result["prediction"]
    best_idx = result["best_idx"]

    q_plot = to_numpy(prediction["q_plot_pred"])
    map_curve = to_numpy(prediction["sampled_curves"])[best_idx]
    sld_x = to_numpy(prediction["sampled_sld_xaxis"])
    map_sld = to_numpy(prediction["sampled_sld_profiles"])[best_idx]

    ax_r, ax_sld = axes[row]
    ax_r.errorbar(curve["q"], curve["r"], yerr=curve["dr"], fmt=".", ms=3, alpha=0.55, label=f"{name} data")
    ax_r.plot(q_plot, map_curve, lw=2, label="NF MAP")
    ax_r.set(xlabel="Q (A^-1)", ylabel="Reflectivity", yscale="log", title=f"{name} reflectivity")
    ax_r.legend()

    ax_sld.plot(sld_x, map_sld, lw=2)
    ax_sld.set(xlabel="Depth (A)", ylabel="SLD (1e-6 A^-2)", title=f"{name} MAP SLD")

plt.show()

## Comments / Observations

The predicted curves appeared capped near `Q = 0.100 A^-1` because `EasyInferenceModel.preprocess_and_sample()` applies reflectorch's high-error-tail filtering by default. That filter removes isolated points with `dR/R >= 0.3` and truncates the curve when three consecutive high-error points occur after `Q >= 0.1`. With the default filter enabled, the effective maximum Q is about `0.1177 A^-1` for `du` and `0.0959 A^-1` for `uu`.

This notebook now sets `ENABLE_ERROR_BARS_FILTERING = False`, so inference uses the full positive-intensity NR range, matching the data treatment in `NR_XRR_Fit_AlO3_Co_Pt.ipynb`: `du` reaches `Q = 0.1883 A^-1`, and `uu` reaches `Q = 0.1850 A^-1` after dropping zero-intensity points.

Compared with the NR_XRR fit notebook, the NF run is only a per-channel smoke test. The fit notebook performs a joint XRR + PNR magnetic fit with shared geometry, magnetic SLD, scale, and background. This NF checkpoint is a standard independent two-layer model, so it does not enforce shared geometry or magnetic coupling between `du` and `uu`.

In the 256-sample full-Q check, `du` is partially consistent with the fit result: Co thickness and Pt/Co roughness are close, while air/Pt and Co/Al2O3 roughness differ. `uu` is poor: the MAP hits the upper thickness bounds for Pt and Co. Treat `uu` results as a model-mismatch signal, not as a reliable magnetic fit.